# 01 · Carregar os dados

Primeiro notebook da ordem. Lê os CSVs brutos do ENEM 2024 uma vez, junta
participantes + redação e grava `enem2024_base_bruta.parquet`. Todos os
notebooks seguintes leem esse arquivo em vez de reler os CSVs.

Rode este antes de qualquer outro. Se os caminhos mudarem, mexe-se só aqui.


## 0 · Caminhos e colunas

In [ ]:
import os
import pandas as pd

DATA_DIR = os.environ.get("ENEM_DIR", "/content")              # onde estão os CSVs brutos
OUT_DIR  = os.environ.get("ENEM_OUT", "data_processed")        # onde gravamos os derivados
os.makedirs(OUT_DIR, exist_ok=True)

PART_PATH = os.path.join(DATA_DIR, "PARTICIPANTES_2024.csv")
RED_PATH  = os.path.join(DATA_DIR, "COMPETENCIAS_REDACAO_2024.csv")
BASE_PATH = os.path.join(OUT_DIR, "enem2024_base_bruta.parquet")

PERFIL   = ["TP_FAIXA_ETARIA","TP_SEXO","TP_ESTADO_CIVIL","TP_COR_RACA","TP_NACIONALIDADE",
            "TP_ST_CONCLUSAO","TP_ANO_CONCLUIU","TP_ENSINO","IN_TREINEIRO","SG_UF_PROVA"]
QUESTOES = [f"Q{i:03d}" for i in range(1, 24)]
CATEG    = PERFIL + QUESTOES

COMP     = [f"NU_NOTA_COMP{k}" for k in range(1, 6)]
AV_NOTA  = ["NU_NOTA_AV1", "NU_NOTA_AV2", "NU_NOTA_AV3", "NU_NOTA_AV4"]
COMP_AV  = [f"NU_NOTA_COMP{k}_AV{a}" for k in range(1, 6) for a in (1, 2)]
RED_COLS = ["NU_INSCRICAO", "TP_STATUS_REDACAO", "NU_NOTA_REDACAO"] + COMP + AV_NOTA + COMP_AV


## 1 · Leitura e merge

`CO_MUNICIPIO_PROVA` entra aqui porque a engenharia de features (notebook 03)
precisa dele; as notas por avaliador entram porque a EDA (02) olha a divergência
entre corretores. Ler a união das colunas uma vez sai mais barato que dois merges.

In [ ]:
part = pd.read_csv(PART_PATH, sep=";", encoding="latin1",
                   usecols=["NU_INSCRICAO", "CO_MUNICIPIO_PROVA"] + CATEG,
                   dtype={c: "category" for c in CATEG})
red  = pd.read_csv(RED_PATH, sep=";", encoding="latin1", usecols=RED_COLS,
                   dtype={c: "float32" for c in RED_COLS if c != "NU_INSCRICAO"})

print("PARTICIPANTES:", part.shape, "| NU_INSCRICAO único:", part["NU_INSCRICAO"].is_unique)
print("REDAÇÃO      :", red.shape, "| NU_INSCRICAO único:", red["NU_INSCRICAO"].is_unique)

df = part.merge(red, on="NU_INSCRICAO", how="outer", validate="one_to_one", indicator=True)
print(df["_merge"].value_counts())
df = df.drop(columns="_merge")
del part, red

print("\nBase unida:", df.shape, f"— {df.memory_usage(deep=True).sum() / 1e9:.2f} GB em memória")
df.head()

## 2 · Gravação

In [ ]:
df.to_parquet(BASE_PATH, index=False)
print(BASE_PATH, f"{os.path.getsize(BASE_PATH) / 1e6:.1f} MB")

## Próximo

`02_analise_exploratoria.ipynb` — lê `enem2024_base_bruta.parquet` e descreve a base.